# 02 - Relaciones implicitas y calidad de datos

Objetivo: validar relaciones que no siempre aparecen como claves foraneas explicitas y medir incidencias relevantes para el DWH.

In [ ]:
from pathlib import Path
import os

import pandas as pd
from sqlalchemy import create_engine, text

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DB_NAME = os.getenv("DB_NAME", "saleshealth")
DB_USER = os.getenv("DB_USER", os.getenv("USER", "postgres"))
DB_HOST = os.getenv("DB_HOST", "")
DB_PORT = os.getenv("DB_PORT", "")
DB_PASSWORD = os.getenv("DB_PASSWORD", "")

if DB_HOST:
    auth = f"{DB_USER}:{DB_PASSWORD}@" if DB_PASSWORD else f"{DB_USER}@"
    port = f":{DB_PORT}" if DB_PORT else ""
    DATABASE_URL = os.getenv("DATABASE_URL", f"postgresql+psycopg2://{auth}{DB_HOST}{port}/{DB_NAME}")
else:
    DATABASE_URL = os.getenv("DATABASE_URL", f"postgresql+psycopg2:///{DB_NAME}")

engine = create_engine(DATABASE_URL)

def q(sql):
    return pd.read_sql_query(text(sql), engine)

In [ ]:
q('''
select
    count(*) as tiendas,
    count(cz.postal_code) as tiendas_con_zona,
    count(*) - count(cz.postal_code) as tiendas_sin_zona
from store s
left join city_zone cz on cz.postal_code = left(s.postal_code, 5);
''')

In [ ]:
q('''
select
    count(*) as devoluciones,
    count(rr.reason_id) as devoluciones_con_motivo,
    count(*) - count(rr.reason_id) as devoluciones_sin_motivo
from return_item ri
left join return_reason rr on rr.reason_id = ri.reason_id;
''')

In [ ]:
q('''
select
    p.product_id,
    p.name as product_name,
    count(si.sale_item_id) as lineas_venta,
    sum(si.quantity) as unidades,
    round(sum(si.subtotal)::numeric, 2) as ingresos,
    cp.unit_cost
from product p
join sale_item si on si.product_id = p.product_id
left join central_product cp on cp.product_id = p.product_id
where cp.unit_cost is null
group by p.product_id, p.name, cp.unit_cost
order by lineas_venta desc;
''')

In [ ]:
q('''
select
    count(*) as lineas,
    count(*) filter (where offer_id is null) as lineas_sin_oferta,
    count(*) filter (where offer_id is not null) as lineas_con_oferta
from sale_item;
''')

Estas comprobaciones justifican decisiones del ETL: registro tecnico `Sin oferta`, registro tecnico `Sin motivo informado`, enriquecimiento por codigo postal y coste imputado trazable.